# Fatores externos e a presença no clube

**Grupo 02 – CCO A2 · SPTech 2026 · Clube de Desbravadores Tamoios**

A chamada do Arandu Digital registra quem esteve presente em cada data. Este notebook monta a outra
metade da pergunta — **o que estava acontecendo na cidade naquele dia** — em três camadas:

| Camada | O que é | Onde fica |
|---|---|---|
| 🥉 **Bronze** | o arquivo exatamente como a fonte publicou, sem uma linha alterada | `bronze/` |
| 🥈 **Silver** | uma tabela por fonte, limpa e tipada, no grão natural dela | `silver/` |
| 🥇 **Gold** | a tabela de análise: uma linha por dia, pronta para juntar com `data_chamada` | `gold/` |

Três fatores, todos de arquivo para download — nenhuma API:

| Fator | Fonte | Atualização | Cobertura |
|---|---|---|---|
| Clima | INMET, estação A701 (SP – Mirante de Santana) | mensal | 2000–2026 |
| Eventos na cidade | MusicBrainz, dump de eventos | 2×/semana | histórico |
| Feriados | ANBIMA, feriados nacionais | estático | 2001–2099 |

### Por que só três

O escopo começou com cinco fatores. **Trânsito** e **futebol** foram cortados por falta de janela, não
por falta de interesse:

- A base de lentidão da **CET-SP** foi descontinuada em 31/12/2023 — o próprio dataset traz o aviso.
- A base do **Brasileirão** na Base dos Dados declara atualização recorrente, mas o arquivo não é
  tocado desde **26/08/2024**, que é também a data do último jogo que ele contém.

As chamadas do clube são de 2026 em diante. Nenhuma das duas jamais cruzaria com elas, e um fator que
entra vazio na análise é pior do que um fator ausente: ele sugere que foi testado. Os três que
sobraram recebem dado novo e cobrem a janela real.

### Três coisas que economizam tempo

1. **O INMET recusa download sem `User-Agent` de navegador** e o **gov.br recusa sem `Referer`.** Nos
   dois casos a conexão volta vazia ou com 403, sem mensagem útil.
2. **O caminho dentro do zip do INMET muda de ano para ano** (2023 sem pasta, 2025 dentro de `2025/`),
   então o membro é procurado pelo nome da estação, não por caminho fixo.
3. **`bronze/` tem ~400 MB** e está no `.gitignore`. `silver/` e `gold/` são pequenos e vão para o Git.

## 🥉 Bronze — ingestão

Baixa e guarda o arquivo como ele veio. Nenhuma transformação: se a fonte publicou com erro, o erro
fica registrado aqui e é tratado na camada seguinte. As células pulam o que já estiver em `bronze/`.

In [1]:
import io, json, re, tarfile, urllib.request, zipfile
from pathlib import Path

import pandas as pd

ANOS = (2023, 2024, 2025, 2026)
ESTACAO = "A701"
FUSO = "America/Sao_Paulo"

BRONZE, SILVER, GOLD = Path("bronze"), Path("silver"), Path("gold")

INMET = "https://portal.inmet.gov.br/uploads/dadoshistoricos"
MUSICBRAINZ = "https://data.metabrainz.org/pub/musicbrainz/data/json-dumps"
ANBIMA = "https://www.anbima.com.br/feriados/arqs/feriados_nacionais.xls"


def buscar(url, referer=None):
    cabecalhos = {"User-Agent": "Mozilla/5.0"}
    if referer:
        cabecalhos["Referer"] = referer
    return urllib.request.urlopen(urllib.request.Request(url, headers=cabecalhos), timeout=600)


def baixar(url, destino, referer=None):
    if not destino.exists():
        destino.parent.mkdir(exist_ok=True)
        print("baixando", destino.name, flush=True)
        with buscar(url, referer) as origem, open(destino, "wb") as fh:
            while bloco := origem.read(1 << 20):
                fh.write(bloco)
    return destino, url


baixados = [baixar(f"{INMET}/{ano}.zip", BRONZE / f"inmet_{ano}.zip") for ano in ANOS]
baixados.append(baixar(f"{MUSICBRAINZ}/{buscar(MUSICBRAINZ + '/LATEST').read().decode().strip()}"
                       "/event.tar.xz", BRONZE / "event.tar.xz"))
baixados.append(baixar(ANBIMA, BRONZE / "feriados_nacionais.xls",
                       referer="https://www.anbima.com.br/feriados/feriados.asp"))

manifesto = pd.DataFrame([{"arquivo": p.name, "mb": round(p.stat().st_size / 1e6, 1), "origem": u}
                          for p, u in baixados])
manifesto.to_csv(BRONZE / "manifesto.csv", index=False)
print("total:", round(manifesto.mb.sum(), 1), "MB")
manifesto[["arquivo", "mb"]]

total: 403.3 MB


,arquivo,mb
0,inmet_2023.zip,107.1
1,inmet_2024.zip,102.8
2,inmet_2025.zip,90.9
3,inmet_2026.zip,55.1
4,event.tar.xz,47.3
5,feriados_nacionais.xls,0.1


## 🥈 Silver — limpeza

Uma tabela por fonte, no grão natural dela. As regras abaixo não são genéricas: cada uma corrige um
problema que existe de fato nesses arquivos, e o número entre parênteses é o que foi medido.

**Clima** — o INMET publica em **UTC**, e a coluna se chama `Hora UTC`. Agrupar por dia sem converter
joga 3 horas de cada dia para o dia errado, o que estraga justamente a chuva do fim da tarde. Aqui o
instante vira `America/Sao_Paulo` antes de qualquer agregação. `-9999` é o código de falha do sensor e
vira nulo (645 leituras de chuva e 212 de temperatura só em 2023).

**Eventos** — fora os 14 cancelados e os 2 sem data completa. Os 15 eventos de mais de um dia são
expandidos para uma linha por dia, o que sozinho leva a cobertura de 132 para 148 dias com evento. O
campo `tipo` é preservado: **metade dos registros não é show** — são 118 Concert, 92 Festival e 22
Convention/Expo, o que inclui coisas como o Anime Friends no Anhembi.

**Feriados** — a planilha da ANBIMA termina com quatro linhas de nota de rodapé no meio da coluna de
data; elas são descartadas por não converterem para data.

In [2]:
def silver_clima():
    partes = []
    for ano in ANOS:
        zip_ano = zipfile.ZipFile(BRONZE / f"inmet_{ano}.zip")
        membro = next(n for n in zip_ano.namelist()
                      if ESTACAO in n and n.upper().endswith(".CSV"))
        with zip_ano.open(membro) as fh:
            tabela = pd.read_csv(fh, sep=";", skiprows=8, encoding="latin-1",
                                 decimal=",", na_values=["-9999", -9999])
        achar = lambda termo: next(c for c in tabela.columns if termo in c.upper())
        tabela = tabela.rename(columns={tabela.columns[0]: "dia", tabela.columns[1]: "hora",
                                        achar("PRECIPITA"): "chuva_mm",
                                        achar("TEMPERATURA DO AR"): "temp_c"})
        hora = tabela["hora"].astype(str).str.replace(r"\D", "", regex=True).str[:4]
        tabela["instante"] = (pd.to_datetime(tabela["dia"] + hora, format="%Y/%m/%d%H%M", utc=True)
                              .dt.tz_convert(FUSO))
        partes.append(tabela[["instante", "chuva_mm", "temp_c"]])

    todas = pd.concat(partes).sort_values("instante").reset_index(drop=True)
    return todas[todas.instante.dt.year.isin(ANOS)]


def silver_eventos():
    registros = []
    with tarfile.open(BRONZE / "event.tar.xz", "r:xz") as tar:
        fluxo = io.TextIOWrapper(tar.extractfile("mbdump/event"), encoding="utf-8")
        for linha in fluxo:
            if "São Paulo" not in linha:
                continue
            evento = json.loads(linha)
            local = next((rel["place"] for rel in evento.get("relations", [])
                          if rel.get("type") == "held at"
                          and ((rel.get("place") or {}).get("area") or {}).get("name") == "São Paulo"),
                         None)
            if local is None or evento.get("cancelled"):
                continue
            periodo = evento.get("life-span") or {}
            inicio, fim = periodo.get("begin") or "", periodo.get("end") or ""
            if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", inicio):
                continue
            if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", fim):
                fim = inicio
            for dia in pd.date_range(inicio, fim, freq="D").date:
                registros.append({"dia": dia, "evento": evento.get("name"),
                                  "local": local.get("name"), "tipo": evento.get("type")})
    return pd.DataFrame(registros).drop_duplicates()


def silver_feriados():
    tabela = pd.read_excel(BRONZE / "feriados_nacionais.xls")
    tabela = tabela[pd.to_datetime(tabela["Data"], errors="coerce").notna()].copy()
    tabela["dia"] = pd.to_datetime(tabela["Data"]).dt.date
    return (tabela[["dia", "Feriado"]].rename(columns={"Feriado": "nome_feriado"})
            .drop_duplicates(subset="dia").sort_values("dia"))


SILVER.mkdir(exist_ok=True)
clima, eventos, feriados = silver_clima(), silver_eventos(), silver_feriados()

for nome, tabela in (("clima_horario", clima), ("eventos_cidade", eventos),
                     ("feriados_nacionais", feriados)):
    tabela.to_csv(SILVER / f"{nome}.csv.gz", index=False)
    print(f"{nome:<20} {len(tabela):>7} linhas")

eventos.tipo.fillna("(sem tipo)").value_counts()

clima_horario          31389 linhas
eventos_cidade           235 linhas
feriados_nacionais      1263 linhas


tipo
Concert            118
Festival            92
Convention/Expo     22
(sem tipo)           2
Launch event         1
Name: count, dtype: int64

## 🥇 Gold — a tabela de análise

Reduz as três tabelas silver ao grão do negócio: **um dia, uma linha**.

`feriado_prolongado` é derivado, não vem de fonte nenhuma: marca os dias que pertencem a um bloco de
**três ou mais dias seguidos** sem expediente, contando fim de semana e feriado juntos. É o que
distingue um feriado que caiu na quarta-feira de um feriadão que esvazia a cidade — e é essa a
diferença que interessa para presença no clube.

In [3]:
def gold_fatores():
    dias = pd.date_range(f"{min(ANOS)}-01-01", f"{max(ANOS)}-12-31", freq="D")
    tabela = pd.DataFrame(index=pd.Index(dias, name="data"))

    por_dia = clima.assign(dia=clima.instante.dt.date)
    tabela = tabela.join(por_dia.groupby("dia").agg(chuva_mm=("chuva_mm", "sum"),
                                                    temp_media_c=("temp_c", "mean"),
                                                    temp_max_c=("temp_c", "max")).round(1))

    tabela = tabela.join(eventos.groupby("dia").size().rename("eventos_sp"))
    tabela["eventos_sp"] = tabela["eventos_sp"].fillna(0).astype(int)

    tabela["nome_feriado"] = tabela.index.map(dict(zip(pd.to_datetime(feriados.dia),
                                                       feriados.nome_feriado)))
    tabela["feriado"] = tabela["nome_feriado"].notna()
    tabela["fim_de_semana"] = tabela.index.dayofweek >= 5

    sem_expediente = tabela["feriado"] | tabela["fim_de_semana"]
    bloco = (sem_expediente != sem_expediente.shift()).cumsum()
    tabela["feriado_prolongado"] = (sem_expediente
                                    & sem_expediente.groupby(bloco).transform("size").ge(3))
    return tabela


def gold_cobertura(fatores):
    registros = []
    for fonte, coluna, tem in (("clima", "temp_media_c", fatores.temp_media_c.notna()),
                               ("eventos", "eventos_sp", fatores.eventos_sp > 0),
                               ("feriados", "feriado", fatores.feriado)):
        dias = fatores.index[tem]
        registros.append({"fonte": fonte,
                          "primeiro_dia": dias.min().date(), "ultimo_dia": dias.max().date(),
                          "dias_com_dado": int(tem.sum()),
                          "cobertura_pct": round(100 * tem.mean(), 1)})
    return pd.DataFrame(registros).set_index("fonte")


GOLD.mkdir(exist_ok=True)
fatores = gold_fatores()
cobertura = gold_cobertura(fatores)

fatores.to_csv(GOLD / "fatores_presenca_dia.csv.gz")
cobertura.to_csv(GOLD / "cobertura_fontes.csv")

print(fatores.shape[0], "dias x", fatores.shape[1], "colunas")
cobertura

1461 dias x 8 colunas


,primeiro_dia,ultimo_dia,dias_com_dado,cobertura_pct
fonte,,,,
clima,2023-01-01,2026-07-31,1307,89.5
eventos,2023-03-07,2026-07-28,78,5.3
feriados,2023-01-01,2026-12-25,51,3.5


## Conferir

A cobertura de clima não chega a 100% de propósito: o INMET publica com cerca de um mês de defasagem,
então o fim do ano corrente ainda está vazio e vai preenchendo a cada atualização.

In [4]:
fatores = pd.read_csv(GOLD / "fatores_presenca_dia.csv.gz", index_col="data", parse_dates=True)

print("período:", fatores.index.min().date(), "a", fatores.index.max().date())
print("dias com chuva:", int((fatores.chuva_mm > 0).sum()))
print("dias com evento na cidade:", int((fatores.eventos_sp > 0).sum()))
print("feriados nacionais:", int(fatores.feriado.sum()),
      "| dias em feriadão:", int(fatores.feriado_prolongado.sum()))
fatores.describe().round(1)

período: 2023-01-01 a 2026-12-31
dias com chuva: 535
dias com evento na cidade: 78
feriados nacionais: 51 | dias em feriadão: 62


,chuva_mm,temp_media_c,temp_max_c,eventos_sp
count,1308.0,1307.0,1307.0,1461.0
mean,4.2,21.0,26.4,0.1
std,10.7,3.4,4.4,0.5
min,0.0,10.7,12.7,0.0
25%,0.0,18.5,23.5,0.0
50%,0.0,21.3,26.8,0.0
75%,2.0,23.4,29.7,0.0
max,125.6,30.4,37.2,5.0


In [5]:
fatores.loc["2026-02-14":"2026-02-17"]

,chuva_mm,temp_media_c,temp_max_c,eventos_sp,nome_feriado,feriado,fim_de_semana,feriado_prolongado
data,,,,,,,,
2026-02-14,0.0,24.2,27.9,0,NaN,False,True,True
2026-02-15,0.0,25.3,32.2,0,NaN,False,True,True
2026-02-16,25.6,24.7,33.5,0,Carnaval,True,False,True
2026-02-17,2.4,25.1,31.4,0,Carnaval,True,False,True


O bloco acima é o Carnaval de 2026: quatro dias seguidos marcados como `feriado_prolongado`, sábado a
terça. Para um clube que se reúne no sábado, é exatamente o tipo de data em que metade da unidade
viajou — e o tipo de linha que a análise precisa distinguir de um sábado comum.

## Como juntar com a chamada

```sql
SELECT c.data_chamada,
       AVG(p.presente) AS percentual_presenca
  FROM Presenca p
  JOIN Chamada  c ON c.idChamada = p.Chamada_idChamada
 GROUP BY c.data_chamada
```

```python
presenca = pd.read_sql(consulta, conexao).set_index("data_chamada")
analise = presenca.join(fatores)
```
